In [1]:
import sys
import numpy as np
import pandas as pd
sys.path.append('../')

# Import the cntmosaic package
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.preprocess import make_full_grid
from cntmosaic.sim import ParticipantGenerator, ContactMatrixGenerator, ContactGenerator
from cntmosaic.vis import plot_mosaic

# Visualisation
import altair as alt

Use the `load_age_distribution` and `load_template_patterns` functions to load the population age distribution and template contact patterns for a given country.

In [2]:
df_age_dist = load_age_distribution('United_States')
patterns = load_template_patterns('United_States')

In [3]:
charts = []
for i, (key, pattern) in enumerate(patterns.items()):
	if i != 0:
		chart = plot_mosaic(pattern, title=key.capitalize(), ylabel=None)
	else:
		chart = plot_mosaic(pattern, title=key.capitalize())
	charts.append(chart)

alt.hconcat(*charts).resolve_scale(color='independent')

alt.HConcatChart(...)

To generate participant data, use the `ParticipantGenerator` class. Pass the number of participants and the age distribution to the constructor. The `generate` method will return a data frame with the generated data.

In [4]:
pg = ParticipantGenerator(n=1000, age_dist=df_age_dist['P'].values)
df_part = pg.generate()

To generate a contact intensity matrix, use the `ContactyMatrixGenerator` class. Pass a dictionary of template patterns (can be obtained using the `load_template_patterns` function) and the age distribution. The `generate` method will return a numpy array of contact intensities.

In [5]:
cmg = ContactMatrixGenerator(patterns, df_age_dist['P'].values)
cint_matrix = cmg.generate()

For example, we can generate 3 different contact patterns.

In [6]:
cint_matrices = [cmg.generate() for _ in range(3)]

In [7]:
pop_level_matrix = np.array(cint_matrices).sum(axis=0)/3

In [8]:
cint_pop = plot_mosaic(pop_level_matrix, title='Population level contact intensity')
cint_sub1 = plot_mosaic(cint_matrices[0], title='Contact intensity: subgroup 1', ylabel=None)
cint_sub2 = plot_mosaic(cint_matrices[1], title='Contact intensity: subgroup 2', ylabel=None)
cint_sub3 = plot_mosaic(cint_matrices[2], title='Contact intensity: subgroup 3', ylabel=None)

alt.hconcat(cint_pop, cint_sub1, cint_sub2, cint_sub3)

alt.HConcatChart(...)

To generate contact data, use the `ContactGenerator` class. Pass the participant data generated by the `ParticipantGenerator` and the contact intensity matrix generated by the `ContactyMatrixGenerator`. The `generate` method will return a data frame with the generated contact data.

In [9]:
cg = ContactGenerator(df_part, cint_matrix)
df_cnt = cg.generate()

## Generate Contact Data for multiple subgroups

In [10]:
df_age_dist_us = load_age_distribution('United_States')
df_age_dist_uk = load_age_distribution('United_Kingdom')

patterns_us = load_template_patterns('United_States')
patterns_uk = load_template_patterns('United_Kingdom')

In [11]:
df_part = ParticipantGenerator(n=[1000, 2000], age_dist=[df_age_dist_us['P'].values, df_age_dist_uk['P'].values]).generate()

In [12]:
cmg_us = ContactMatrixGenerator(patterns_us, df_age_dist_us['P'].values)
cmg_uk = ContactMatrixGenerator(patterns_uk, df_age_dist_uk['P'].values)

cint_matrices = [cmg_us.generate(), cmg_uk.generate()]

In [13]:
chart_us = plot_mosaic(cint_matrices[0], title='Contact intensity: United States')
chart_uk = plot_mosaic(cint_matrices[1], title='Contact intensity: United Kingdom')

alt.hconcat(chart_us, chart_uk).resolve_scale(color='independent')

alt.HConcatChart(...)

In [14]:
df_cnt = ContactGenerator(df_part, cint_matrices).generate()